In [29]:
import os, glob
import cv2
import numpy as np
import matplotlib.pyplot as plt

from skimage.util import view_as_blocks
from sklearn.mixture import GaussianMixture
from sklearn.metrics import precision_score, recall_score, f1_score


In [30]:
ROOT = r"D:\University\Machine Learning\Dataset\Segmentation"

ORIGINAL_FOLDER = os.path.join(ROOT, r"D:\University\Machine Learning\Dataset\Segmentation\Original Image")
GT_FOLDER = os.path.join(ROOT, r"D:\University\Machine Learning\Dataset\Segmentation\Ground Truth")
PRE_FOLDER = os.path.join(ROOT, r"D:\University\Machine Learning\Dataset\Segmentation\Preprocessed")
GT_BLOCK_FOLDER = os.path.join(ROOT, r"D:\University\Machine Learning\Dataset\Segmentation\GroundTruthBlock")

os.makedirs(PRE_FOLDER, exist_ok=True)
os.makedirs(GT_BLOCK_FOLDER, exist_ok=True)


In [31]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def morphological_preprocessing(img_gray, radius=3):

    # tạo kernel ellipse
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (2*radius+1, 2*radius+1)
    )

    # bước 1: erosion
    eroded = cv2.erode(img_gray, kernel)

    # bước 2: opening
    opened = cv2.morphologyEx(eroded, cv2.MORPH_OPEN, kernel)

    # bước 3: erosion cuối
    smoothed = cv2.erode(opened, kernel)

    return smoothed


In [32]:
img_paths = glob.glob(os.path.join(ORIGINAL_FOLDER, "*.jpg"))
print("Preprocessing:", len(img_paths), "images")

for path in img_paths:
    img = cv2.imread(path)
    name = os.path.basename(path)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    norm = morphological_preprocessing(gray,radius=3)
    median = cv2.medianBlur(norm, 9)

    cv2.imwrite(os.path.join(PRE_FOLDER, name), median)

print("Done preprocessing.")



Preprocessing: 130 images
Done preprocessing.


In [33]:
BLOCK_SIZE = (150, 150)

def extract_blocks(img, block_size):
    h, w = img.shape
    bh, bw = block_size

    new_h = (h // bh) * bh
    new_w = (w // bw) * bw
    img = img[:new_h, :new_w]

    blocks = view_as_blocks(img, block_size)
    return blocks


In [34]:
THRESHOLD_PIXEL = 150  # paper-style

gt_paths = glob.glob(os.path.join(GT_FOLDER, "*.png"))

for path in gt_paths:
    gt = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    name = os.path.basename(path)

    blocks = extract_blocks(gt, BLOCK_SIZE)
    crack_pixels = np.sum(blocks > 0, axis=(2, 3))
    gt_block = (crack_pixels > THRESHOLD_PIXEL).astype(np.uint8)

    np.save(
        os.path.join(GT_BLOCK_FOLDER, name.replace(".png", ".npy")),
        gt_block
    )

print("Done GT block labeling.")


Done GT block labeling.


In [35]:
def compute_block_mean(blocks):
    H, W = blocks.shape[:2]
    M = np.zeros((H, W))
    for i in range(H):
        for j in range(W):
            M[i, j] = np.mean(blocks[i, j])
    return M


In [36]:
def preliminary_labeling_paper(M, k1=1.0, k2=0.5):
    H, W = M.shape
    plb = np.zeros((H, W), dtype=np.uint8)

    # Horizontal
    for j in range(W - 1):
        Bh = M[:, j]
        th_h = k1 * np.std(Bh) + k2 * np.mean(Bh)
        for i in range(H):
            Ah = np.array([M[i, j], M[i, j+1]])
            if np.std(Ah) > th_h and (Ah[0] - Ah[1]) > 0:
                plb[i, j] = 1

    # Vertical
    for i in range(H - 1):
        Bv = M[i, :]
        th_v = k1 * np.std(Bv) + k2 * np.mean(Bv)
        for j in range(W):
            Av = np.array([M[i, j], M[i+1, j]])
            if np.std(Av) > th_v and (Av[0] - Av[1]) > 0:
                plb[i, j] = 1

    return plb


In [37]:
X_all = []

for path in glob.glob(os.path.join(PRE_FOLDER, "*.jpg")):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    blocks = extract_blocks(img, BLOCK_SIZE)
    M = compute_block_mean(blocks)
    X_all.append(M.flatten())

X_all = np.concatenate(X_all).reshape(-1, 1)

gmm = GaussianMixture(n_components=2, covariance_type="full", random_state=0)
gmm.fit(X_all)


GaussianMixture(n_components=2, random_state=0)

In [38]:
def detect_block_mask(img, gmm):
    blocks = extract_blocks(img, BLOCK_SIZE)
    M = compute_block_mean(blocks)
    X = M.flatten().reshape(-1, 1)

    labels = gmm.predict(X)
    crack_label = np.argmin(gmm.means_)  # darker = crack

    mask = (labels == crack_label).astype(np.uint8)
    return mask.reshape(M.shape)


In [39]:
def compute_fmeasure(gt_block, pred_block):
    y_true = gt_block.flatten()
    y_pred = pred_block.flatten()

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    return precision, recall, f1


In [47]:
idx = 37
img_path = glob.glob(os.path.join(PRE_FOLDER, "*.jpg"))[idx]
name = os.path.basename(img_path)

img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
gt_block = np.load(os.path.join(
    GT_BLOCK_FOLDER, name.replace(".jpg", ".npy")
))

pred_block = detect_block_mask(img, gmm)

pr, re, f1 = compute_fmeasure(gt_block, pred_block)

print("Image:", name)
print("Precision:", pr)
print("Recall:", re)
print("F1:", f1)


Image: 15.jpg
Precision: 0.05764411027568922
Recall: 0.9583333333333334
F1: 0.10874704491725769
